In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

url = "https://www.sanidad.gob.es/profesionales/nomenclator.do?metodo=nomenclatorExcel"     #es el link de decarga del excel directamente,en vez del del enunciado
response = requests.get(url)

with open("../data/nomenclator.xls","wb") as archivo:  #abre(en este caso como no existe,lo crea) un archivo en esa ruta."wb" es que es en modo "write" y "binary",es decir,vamos a meter en este archivo bytes tal cual,no texto,y no lo stoques ni interpretes
    archivo.write(response.content)     #usamos response content porque son bytes,antes hemos usado response.text porque eran letras
print("archivo excel abierto/creado")

In [ ]:
df_precios = pd.read_excel("../data/nomenclator.xls")
df_precios.columns      #para ver el nommbre de todas las columnas

Index(['Código Nacional', 'Nombre del producto farmacéutico',
       'Tipo de fármaco', 'Nombre genérico efecto y accesorio',
       'Código del laboratorio ofertante', 'Nombre del laboratorio ofertante',
       'Estado', 'Fecha de alta en el nomenclátor',
       'Fecha de baja en el nomenclátor', 'Aportación del beneficiario',
       'Principio activo o asociación de principios activos',
       'Precio de venta al público con IVA', 'Precio de referencia',
       'Menor precio de la agrupación homogéna del producto sanitario',
       'Código de la agrupación homegénea del productor sanitario',
       'Nombre de la agrupación homogénea del producto sanitario',
       'Diagnóstico hospitalario', 'Tratamiento de larga duración',
       'Especial control médico', 'Medicamento huérfano'],
      dtype='object')

In [5]:
df_precios_columnas_añadir = df_precios[["Código Nacional", "Estado", "Precio de venta al público con IVA","Precio de referencia","Tratamiento de larga duración", "Especial control médico" ]]

In [7]:
df_hu2 = pd.read_excel("../data/HU2_medicamentos_diabetes.xlsx")
df_hu2.columns

Index(['nregistro', 'nombre', 'pactivos', 'labtitular', 'labcomercializador',
       'cn', 'dosis', 'forma_farmaceutica_simplificada', 'estado_aut',
       'estado_rev', 'comercializado', 'requiere_receta', 'generico',
       'afecta_conduccion', 'triangulo_negro', 'medicamento_huerfano',
       'biosimilar', 'url_html_ficha_tecnica', 'url_foto_materiales',
       'num_registros_atc', 'num_principios_activos', 'num_excipientes',
       'vias_administracion', 'palabras_seccion_4_4', 'numero_tablas',
       'indicador_riesgo_severo'],
      dtype='object')

In [ ]:
print(df_hu2["cn"].dtype)
print(df_hu2["cn"].head())

print(df_precios_columnas_añadir["Código Nacional"].dtype)
print(df_precios_columnas_añadir["Código Nacional"].head())
#Vemos que los valores que queremos unir tienen formatos diferentes,uno es float y otro int

float64
0    662260.0
1    662258.0
2    775536.0
3    775502.0
4    764777.0
Name: cn, dtype: float64
int64
0    140001
1    140002
2    140003
3    140004
4    140005
Name: Código Nacional, dtype: int64


In [10]:
df_hu2["cn"].isna().sum()

np.int64(1)

In [11]:
df_precios_columnas_añadir = df_precios[["Código Nacional", "Estado", "Precio de venta al público con IVA","Precio de referencia","Tratamiento de larga duración", "Especial control médico" ]]
# IMPORTANTE: doble corchete = selecciono varias columnas a la vez (lista de nombres)del excel df_precios y me sigue devolviendo una tabla, no una sola columna
df_precios_columnas_añadir["Código Nacional"] = df_precios_columnas_añadir["Código Nacional"].astype("float64")
#tenemos que convertir la columna de Codigo Nacional del excel nuevo a float(son int),ya que la columna "cn" de nuestro excel del HU2 esta en float

df_hu3 = df_hu2.merge(df_precios_columnas_añadir, left_on="cn", right_on="Código Nacional",how="left")
#cojo mi tabla principal(df_hu2,es la que tiene todo hasta ahora) y le pego las columnas de la ptra tabla (df_precios_columnas_añadir).Con how=left nos quedamos con todas las filas del de la izq pase lo que pase(df_hu2)

print(df_hu3.shape)
print(df_hu3.columns)

(1569, 32)
Index(['nregistro', 'nombre', 'pactivos', 'labtitular', 'labcomercializador',
       'cn', 'dosis', 'forma_farmaceutica_simplificada', 'estado_aut',
       'estado_rev', 'comercializado', 'requiere_receta', 'generico',
       'afecta_conduccion', 'triangulo_negro', 'medicamento_huerfano',
       'biosimilar', 'url_html_ficha_tecnica', 'url_foto_materiales',
       'num_registros_atc', 'num_principios_activos', 'num_excipientes',
       'vias_administracion', 'palabras_seccion_4_4', 'numero_tablas',
       'indicador_riesgo_severo', 'Código Nacional', 'Estado',
       'Precio de venta al público con IVA', 'Precio de referencia',
       'Tratamiento de larga duración', 'Especial control médico'],
      dtype='object')


/var/folders/8d/zpcwywvx609fvns0pq_b9b4r0000gn/T/ipykernel_243/2241907899.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_precios_columnas_añadir["Código Nacional"] = df_precios_columnas_añadir["Código Nacional"].astype("float64")


In [12]:
df_hu3["Estado"].isna().sum()

np.int64(742)

In [13]:
df_precios_columnas_añadir["Código Nacional"].dtype

dtype('float64')

In [14]:
df_hu2["comercializado"].value_counts()

comercializado
0    790
1    779
Name: count, dtype: int64

In [15]:
pd.crosstab(df_hu2["comercializado"], df_hu3["Estado"].isna())

Estado,False,True
comercializado,,
0,158,632
1,669,110


In [19]:
df_hu3.to_excel("../data/HU3_medicamentos_diabetes.xlsx",index=False)


In [20]:
comprobacion = pd.read_excel("../data/HU3_medicamentos_diabetes.xlsx")
comprobacion.shape

(1569, 32)